# E071 -- Dark Matter: Galaxy Rotation Curves (SPARC)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e071_dark_matter.ipynb)

**Objective:** Download the SPARC (Spitzer Photometry and Accurate Rotation Curves) database of 175 late-type galaxies, analyze the flat rotation curve problem, compute mass discrepancies, and quantify the dark matter fraction across the galaxy population.

The flat rotation curve problem is the single strongest piece of evidence for dark matter: galaxies rotate too fast at large radii for the visible matter alone to hold them together. Either there is invisible mass (dark matter) or gravity works differently at galactic scales.

## Data Source

- **Database:** SPARC — Spitzer Photometry and Accurate Rotation Curves
- **Reference:** Lelli, McGaugh, & Schombert (2016), AJ 152, 157
- **URL:** Zenodo archive of rotation curve .dat files
- **Format:** Tab-separated .dat files with header comments
- **Columns:** Rad (kpc), Vobs (km/s), errV, Vgas, Vdisk, Vbul, SBdisk, SBbul
- **License:** Public (academic)

In [ ]:
import urllib.request
import zipfile
import io
import os
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Download SPARC rotation curves from Zenodo
url = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip"
print("Downloading SPARC rotation curves from Zenodo...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=120)
zip_data = io.BytesIO(response.read())
print(f"Downloaded {zip_data.getbuffer().nbytes / 1e6:.1f} MB")

# Extract all .dat files
galaxies = {}
with zipfile.ZipFile(zip_data) as zf:
    dat_files = [f for f in zf.namelist() if f.endswith('.dat')]
    print(f"Found {len(dat_files)} .dat files")
    
    for fname in dat_files:
        with zf.open(fname) as f:
            content = f.read().decode('utf-8', errors='replace')
        
        gal_name = os.path.basename(fname).replace('_rotmod.dat', '').replace('.dat', '')
        
        rows = []
        for line in content.split('\n'):
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('!'):
                continue
            parts = line.split()
            if len(parts) >= 6:
                try:
                    vals = [float(x) for x in parts[:8] if x.replace('.', '').replace('-', '').replace('+', '').replace('e', '').isdigit() or '.' in x]
                    if len(vals) >= 6:
                        rows.append(vals[:8] if len(vals) >= 8 else vals + [0.0] * (8 - len(vals)))
                except ValueError:
                    continue
        
        if len(rows) >= 3:
            arr = np.array(rows)
            galaxies[gal_name] = {
                'Rad': arr[:, 0],     # kpc
                'Vobs': arr[:, 1],    # km/s
                'errV': arr[:, 2],    # km/s
                'Vgas': arr[:, 3],    # km/s
                'Vdisk': arr[:, 4],   # km/s
                'Vbul': arr[:, 5],    # km/s
            }

print(f"\nParsed {len(galaxies)} galaxies with valid rotation curves")

## Flat Rotation Curves

In Newtonian gravity, orbital velocity should fall as v ~ 1/sqrt(r) beyond the visible disk (like planets in the solar system). Instead, galaxy rotation curves stay **flat** — constant velocity out to the largest measured radii. This requires either dark matter or modified gravity.

In [ ]:
# Analyze flatness: for each galaxy, compute the ratio of outer to peak velocity
flat_ratios = []
gal_names_sorted = []
peak_vels = []
outer_vels = []
max_radii = []

for name, g in galaxies.items():
    V = g['Vobs']
    R = g['Rad']
    if len(V) < 5 or V.max() < 20:
        continue
    
    v_peak = V.max()
    # Outer velocity: mean of last 3 points
    v_outer = V[-3:].mean()
    ratio = v_outer / v_peak
    
    flat_ratios.append(ratio)
    gal_names_sorted.append(name)
    peak_vels.append(v_peak)
    outer_vels.append(v_outer)
    max_radii.append(R.max())

flat_ratios = np.array(flat_ratios)
peak_vels = np.array(peak_vels)
outer_vels = np.array(outer_vels)
max_radii = np.array(max_radii)

print(f"=== Rotation Curve Flatness ===")
print(f"  Galaxies analyzed: {len(flat_ratios)}")
print(f"  V_outer / V_peak: mean = {flat_ratios.mean():.3f}, median = {np.median(flat_ratios):.3f}")
print(f"  Fraction with ratio > 0.8 (flat): {(flat_ratios > 0.8).mean()*100:.1f}%")
print(f"  Keplerian expectation at 2x peak radius: ~0.71 (1/sqrt(2))")
print(f"\n  Peak velocity range: [{peak_vels.min():.0f}, {peak_vels.max():.0f}] km/s")
print(f"  Max radius range: [{max_radii.min():.1f}, {max_radii.max():.1f}] kpc")

In [ ]:
# Mass discrepancy: D = Vobs^2 / Vbar^2 where Vbar^2 = Vgas^2 + Vdisk^2 + Vbul^2
all_r_norm = []   # R / R_max
all_D = []        # mass discrepancy
all_Vobs = []
all_Vbar = []
dm_fractions = []  # dark matter fraction at outermost point

for name, g in galaxies.items():
    V = g['Vobs']
    R = g['Rad']
    if len(V) < 5 or V.max() < 20:
        continue
    
    Vbar2 = g['Vgas']**2 + g['Vdisk']**2 + g['Vbul']**2
    Vbar = np.sqrt(np.maximum(Vbar2, 1.0))  # avoid sqrt of negative
    
    D = V**2 / np.maximum(Vbar2, 1.0)
    
    r_norm = R / R.max()
    all_r_norm.extend(r_norm)
    all_D.extend(D)
    all_Vobs.extend(V)
    all_Vbar.extend(Vbar)
    
    # Dark matter fraction at outer point
    dm_frac = 1 - 1.0 / max(D[-1], 1.0)
    dm_fractions.append(max(0, dm_frac))

all_r_norm = np.array(all_r_norm)
all_D = np.array(all_D)
all_Vobs = np.array(all_Vobs)
all_Vbar = np.array(all_Vbar)
dm_fractions = np.array(dm_fractions)

print(f"=== Mass Discrepancy ===")
print(f"  Total data points across all galaxies: {len(all_D)}")
print(f"  Median mass discrepancy D: {np.median(all_D):.2f}")
print(f"  D at outer radii (r/R_max > 0.8): {np.median(all_D[all_r_norm > 0.8]):.2f}")
print(f"  D = 1 means no dark matter needed; D >> 1 means dark matter dominates")
print(f"\n=== Dark Matter Fraction (outer radius) ===")
print(f"  Mean: {dm_fractions.mean()*100:.1f}%")
print(f"  Median: {np.median(dm_fractions)*100:.1f}%")

In [ ]:
# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E071: Dark Matter — Galaxy Rotation Curves (SPARC)",
             fontsize=15, fontweight='bold')

# (a) Sample rotation curves (pick 6 diverse galaxies)
ax = axes[0, 0]
# Sort by peak velocity, pick evenly spaced
sorted_gals = sorted(galaxies.items(), key=lambda x: x[1]['Vobs'].max())
n_gals = len(sorted_gals)
sample_idx = np.linspace(0, n_gals - 1, 6, dtype=int)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))

for idx, color in zip(sample_idx, colors):
    name, g = sorted_gals[idx]
    ax.errorbar(g['Rad'], g['Vobs'], yerr=g['errV'], fmt='-o', markersize=3,
                color=color, linewidth=1.5, capsize=2, label=name[:12])
    # Plot baryonic contribution
    Vbar = np.sqrt(g['Vgas']**2 + g['Vdisk']**2 + g['Vbul']**2)
    ax.plot(g['Rad'], Vbar, '--', color=color, alpha=0.5, linewidth=1)

ax.set_xlabel('Radius [kpc]', fontsize=11)
ax.set_ylabel('Rotation Velocity [km/s]', fontsize=11)
ax.set_title('(a) Rotation Curves (solid=obs, dashed=baryonic)', fontsize=12)
ax.legend(fontsize=8, ncol=2)

# (b) Mass discrepancy vs normalized radius
ax = axes[0, 1]
ax.scatter(all_r_norm, all_D, s=1, alpha=0.05, color='steelblue')
# Binned median
r_edges = np.linspace(0, 1, 20)
r_mids, D_meds = [], []
for i in range(len(r_edges) - 1):
    m = (all_r_norm >= r_edges[i]) & (all_r_norm < r_edges[i + 1])
    if m.sum() > 10:
        r_mids.append((r_edges[i] + r_edges[i + 1]) / 2)
        D_meds.append(np.median(all_D[m]))
ax.plot(r_mids, D_meds, 'r-o', markersize=5, linewidth=2, label='Median')
ax.axhline(1, color='green', linestyle='--', linewidth=1.5, label='D=1 (no DM)')
ax.set_xlabel('Normalized Radius (R / R_max)', fontsize=11)
ax.set_ylabel('Mass Discrepancy D = V$_{obs}^2$ / V$_{bar}^2$', fontsize=11)
ax.set_title('(b) Mass Discrepancy vs Radius', fontsize=12)
ax.set_ylim(0, min(20, np.percentile(all_D, 99)))
ax.legend(fontsize=10)

# (c) Dark matter fraction histogram
ax = axes[1, 0]
ax.hist(dm_fractions * 100, bins=25, color='mediumpurple', edgecolor='black',
        linewidth=0.5, alpha=0.8)
ax.axvline(np.median(dm_fractions) * 100, color='red', linestyle='--', linewidth=2,
           label=f'Median = {np.median(dm_fractions)*100:.0f}%')
ax.set_xlabel('Dark Matter Fraction at Outer Radius [%]', fontsize=11)
ax.set_ylabel('Number of Galaxies', fontsize=11)
ax.set_title('(c) Dark Matter Fraction Distribution', fontsize=12)
ax.legend(fontsize=10)

# (d) Radial Acceleration Relation (RAR)
ax = axes[1, 1]
# g_obs = Vobs^2 / R, g_bar = Vbar^2 / R
all_R = []
for name, g in galaxies.items():
    if len(g['Vobs']) >= 5 and g['Vobs'].max() >= 20:
        all_R.extend(g['Rad'])
all_R = np.array(all_R)
valid_r = all_R > 0
g_obs = (all_Vobs[valid_r] * 1e3)**2 / (all_R[valid_r] * 3.086e19)  # m/s^2
g_bar = (all_Vbar[valid_r] * 1e3)**2 / (all_R[valid_r] * 3.086e19)

pos = (g_obs > 0) & (g_bar > 0)
ax.scatter(np.log10(g_bar[pos]), np.log10(g_obs[pos]), s=1, alpha=0.05, color='teal')
lims = np.array([g_bar[pos].min(), g_bar[pos].max()])
ax.plot(np.log10(lims), np.log10(lims), 'k--', linewidth=1.5, label='1:1 (no DM)')
ax.set_xlabel('log$_{10}$(g$_{bar}$) [m/s$^2$]', fontsize=11)
ax.set_ylabel('log$_{10}$(g$_{obs}$) [m/s$^2$]', fontsize=11)
ax.set_title('(d) Radial Acceleration Relation', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('e071_dark_matter.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e071_dark_matter.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Galaxies analyzed | ~175 from SPARC database |
| Rotation curves are flat | V_outer/V_peak >> Keplerian expectation |
| Mass discrepancy at outer radii | D >> 1 (dark matter dominates) |
| Dark matter fraction (outer) | Median ~70-80% |
| Radial Acceleration Relation | Tight correlation between g_obs and g_bar |

**Physical interpretation:** Galaxy rotation curves provide the most direct evidence for dark matter. The observed velocities far exceed what visible baryonic matter (stars + gas) can explain. The mass discrepancy increases with radius, showing that dark matter halos extend far beyond the visible galaxy. The tight Radial Acceleration Relation (McGaugh et al. 2016) shows that the dark matter distribution is strongly correlated with baryonic matter, suggesting a deep connection between the two — either through dark matter halo properties or modified gravity.